In [36]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [37]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID" # Ensures IDs match nvidia-smi
os.environ["CUDA_VISIBLE_DEVICES"] = "7"

import random
from pathlib import Path

import numpy as np
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torchvision
from torchvision.transforms import transforms

from PIL import Image

from preprocessing_module import find_class_names_filenames, stratified_split_data_paths, NatureCityScenesDataset
from trainer_class import Trainer

from ResNet import ResNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def plot_metrics(class_names, num_epochs, results_dict, title_str, savefile=None):

    epochs_list = np.arange(1, num_epochs+1)

    plt.figure(figsize=(12, 10))

    plt.subplot(2, 2, 1)
    plt.plot(epochs_list, results_dict["train_loss"], '.-', label="train loss")
    plt.plot(epochs_list, results_dict["val_loss"], '.-', label="val loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()    

    plt.subplot(2, 2, 2)
    plt.plot(epochs_list, results_dict["train_acc"], '.-', label="train acc")
    plt.plot(epochs_list, results_dict["val_acc"], '.-', label="val acc")
    plt.plot(epochs_list, results_dict["val_mAP"], '.-', label="val mAP")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()

    plt.subplot(2, 2, 3) 
    hist_arr_val_acc_per_class = np.asarray(results_dict["val_acc_per_class"])
    for class_number, class_name in enumerate(class_names):
        plt.plot(epochs_list, hist_arr_val_acc_per_class[:, class_number], ".-", label=f"Class {class_number}: {class_name}")

    plt.xlabel("Epoch")
    plt.ylabel("Accuracy, validation set")
    plt.legend()


    plt.subplot(2, 2, 4) 
    hist_arr_val_AP_per_class = np.asarray(results_dict["val_AP_per_class"])
    for class_number, class_name in enumerate(class_names):
        plt.plot(epochs_list, hist_arr_val_AP_per_class[:, class_number], ".-", label=f"Class {class_number}: {class_name}")

    plt.xlabel("Epoch")
    plt.ylabel("Average precision, validation set")
    plt.legend()
     
    plt.suptitle(title_str, fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])

    if savefile:
        plt.savefig(savefile, bbox_inches="tight")
    
    plt.show()

Device: cuda


In [38]:
set_seed(42)

DATASET_PATH = r"/itf-fi-ml/shared/courses/IN3310/mandatory1_data"

class_names, class_filenames = find_class_names_filenames(DATASET_PATH)
x_train_paths, x_val_paths, x_test_paths, y_train, y_val, y_test = stratified_split_data_paths(class_names, class_filenames)

transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_set = NatureCityScenesDataset(DATASET_PATH, x_train_paths, y_train, transform=transform)
val_set = NatureCityScenesDataset(DATASET_PATH, x_val_paths, y_val, transform=transform)
test_set = NatureCityScenesDataset(DATASET_PATH, x_test_paths, y_test, transform=transform)

# --------------------------------------------------------------------------------------------------------------------------------

BATCH_SIZE = 64
train_loader = torch.utils.data.DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = torch.utils.data.DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)

def batch_noise_augment(images: torch.Tensor, std: float = 0.03) -> torch.Tensor:
    noise = torch.randn_like(images) * std
    return torch.clamp(images + noise, -1.0, 1.0)

# --------------------------------------------------------------------------------------------------------------------------------


In [39]:
model_ResNet = ResNet(img_channels=3, num_layers=18, num_classes=6).to(device)

num_epochs = 8

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_ResNet.parameters(), lr=1e-4)

trainer_ResNet = Trainer(
    model=model_ResNet,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    run_name="ResNet",
    augment_fn=lambda x: batch_noise_augment(x, std=0.02),
)

trainer_ResNet.fit(train_loader, val_loader, epochs=num_epochs)

trainer_ResNet.evaluate(test_loader)

# Move results from GPU to memory
results_dict = trainer_ResNet.history

Epoch 01 | train loss: 0.7398, acc: 0.722 | val loss: 0.7430, acc: 0.733, acc_per_class: [0.5506329  0.9665654  0.8788732  0.68406594 0.51197606 0.7919075 ], mean average precision (mAP): 0.845, AP per class: [0.81323294 0.97112087 0.83289056 0.78844216 0.80959555 0.85614258]
Epoch 02 | train loss: 0.4793, acc: 0.823 | val loss: 0.4884, acc: 0.826, acc_per_class: [0.79113925 0.94832826 0.72676057 0.72527474 0.9101796  0.8699422 ], mean average precision (mAP): 0.906, AP per class: [0.90534811 0.98455071 0.86679769 0.85905222 0.90285663 0.91675655]
Epoch 03 | train loss: 0.3815, acc: 0.863 | val loss: 0.6490, acc: 0.777, acc_per_class: [0.73417723 0.99088144 0.6169014  0.75       0.8712575  0.71387285], mean average precision (mAP): 0.876, AP per class: [0.83793009 0.97781421 0.84715669 0.83781967 0.90773229 0.84938259]
Epoch 04 | train loss: 0.2740, acc: 0.907 | val loss: 0.6663, acc: 0.784, acc_per_class: [0.8607595  0.94832826 0.50140846 0.8296703  0.8083832  0.77745664], mean averag